In [ ]:
import sys
sys.path.insert(0, "..")

import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from src.pose_estimation.dataset.real_world import RealWorldDataset
from wicompass.visualization import plot_poses_grid, BONE_CONNECTIONS

%matplotlib inline

# Dataset configuration
config = {
    "dataset_path": "../datasets/real_world/full_real_world_exp/baseline",
    "data_format": {
        "merge_nframes": 3,
        "n_points": 200,
        "normalized": True,
        "radar_input_c": 5,
        "num_joints": 22,
    },
}

# Load dataset
dataset = RealWorldDataset(config, split="train", device="cuda")
print(f"Dataset info: {dataset.get_data_info()}")


In [ ]:
# Load 10 samples (radar + pose)
n_samples = 10
indices = np.linspace(0, len(dataset) - 1, n_samples, dtype=int)

radars, poses = [], []
for idx in indices:
    radar, label = dataset[idx]
    radars.append(radar.cpu().numpy())
    poses.append(label.cpu().numpy())

radars = np.array(radars)  # (10, 200, 5)
poses = np.array(poses)    # (10, 22, 3)
titles = [f"Sample {i}" for i in indices]
print(f"Loaded {n_samples} samples: radar={radars.shape}, pose={poses.shape}")


In [ ]:
# Visualize 10 poses in grid
fig = plot_poses_grid(poses, titles=titles, n_cols=5, show_axes=True)
plt.show()


In [ ]:
# Visualize radar point cloud + skeleton (interactive 3D)
def vis_radar_pose(radar, pose, title=""):
    """Visualize radar point cloud and pose skeleton together."""
    # Filter zero-padded points
    valid = ~np.all(radar[:, :3] == 0, axis=1)
    radar = radar[valid]
    
    fig = go.Figure()
    # Radar points (color by intensity)
    fig.add_trace(go.Scatter3d(
        x=radar[:, 0], y=radar[:, 1], z=radar[:, 2], mode='markers',
        marker=dict(size=3, color=radar[:, 3], colorscale='Viridis', opacity=0.7),
        name=f'Radar ({len(radar)} pts)'
    ))
    # Skeleton joints
    fig.add_trace(go.Scatter3d(
        x=pose[:, 0], y=pose[:, 1], z=pose[:, 2], mode='markers',
        marker=dict(size=5, color='red'), name='Joints'
    ))
    # Skeleton bones
    for i, j in BONE_CONNECTIONS:
        fig.add_trace(go.Scatter3d(
            x=[pose[i, 0], pose[j, 0]], y=[pose[i, 1], pose[j, 1]], z=[pose[i, 2], pose[j, 2]],
            mode='lines', line=dict(color='red', width=3), showlegend=False
        ))
    fig.update_layout(title=title, width=700, height=500,
                      scene=dict(aspectmode='data'), margin=dict(l=0, r=0, t=30, b=0))
    return fig

# Show first sample
vis_radar_pose(radars[0], poses[0], titles[0]).show()


In [ ]:
# Visualize all 10 samples in grid (radar + skeleton)
fig = make_subplots(rows=2, cols=5, specs=[[{'type': 'scatter3d'}]*5]*2,
                    subplot_titles=titles, horizontal_spacing=0.02, vertical_spacing=0.05)

for i, (radar, pose, title) in enumerate(zip(radars, poses, titles)):
    row, col = i // 5 + 1, i % 5 + 1
    valid = ~np.all(radar[:, :3] == 0, axis=1)
    radar = radar[valid]
    
    # Radar points
    fig.add_trace(go.Scatter3d(
        x=radar[:, 0], y=radar[:, 1], z=radar[:, 2], mode='markers',
        marker=dict(size=2, color=radar[:, 3], colorscale='Viridis', opacity=0.6),
        showlegend=False
    ), row=row, col=col)
    # Skeleton
    fig.add_trace(go.Scatter3d(
        x=pose[:, 0], y=pose[:, 1], z=pose[:, 2], mode='markers',
        marker=dict(size=3, color='red'), showlegend=False
    ), row=row, col=col)
    for j1, j2 in BONE_CONNECTIONS:
        fig.add_trace(go.Scatter3d(
            x=[pose[j1, 0], pose[j2, 0]], y=[pose[j1, 1], pose[j2, 1]], z=[pose[j1, 2], pose[j2, 2]],
            mode='lines', line=dict(color='red', width=2), showlegend=False
        ), row=row, col=col)

fig.update_layout(height=600, width=1400, title_text="Radar Point Cloud + Pose Skeleton")
fig.show()
